# Temporal Difference Prediction

# TD Control Task : State Action Reward State Action (SARSA)


In [20]:
import gym
import random
import pandas as pd

In [21]:
#Now, we create the Frozen Lake environment using Gym:
env = gym.make('FrozenLake-v1')

/usr/local/lib/python3.13/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.13/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


In [22]:
#Initialize the discount factor 𝛾, the learning rate 𝛼,
alpha = 0.85
gamma = 0.90

#and the epsilon value:
epsilon = 0.8

In [23]:
#Set the number of episodes and number of time steps in the episode:
num_episodes = 50000
num_timesteps = 1000

In [24]:
#define a random policy to choose random actions
def random_policy():
  return env.action_space.sample()

In [25]:
env.action_space
#Displays count of number of actions : left, right, top, down

Discrete(4)

# TD Prediction

In [26]:
#Initialising the state values to zero.
#Create a empty dictionary : for storing the value of states (16 states, 4x4 grid)
v = {}
for s in range (env.observation_space.n):
  #Cannot make the observation space discrete, since it is 5x5 continous space
  v[s]=0.0

In [27]:
# Patch the NumPy compatibility issue dynamically
import numpy as np
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

#Compute the policy
#For each episode:
for i in range(num_episodes): #50000
#Initialize the state by resetting the environment:
    s = env.reset() #reset the frozenlake environment
#For each step in the episode:
    for t in range(num_timesteps): #1000
# Select an action according to the random policy
        a = random_policy()
#Perform the selected action and store the next state information:
        s_, r, done, _ = env.step(a)
#Compute the value of state as
        v[s] += alpha * (r + gamma * v[s_]-v[s]) # TD update rule
        #print(v[(s)])
        s = s_
        #a = a_
#If the current state is the terminal state, then break:
        if done:
            break

In [28]:
df = pd.DataFrame(list(v.items()), columns=['state', 'value'])

In [29]:
df

,state,value
0,0,0.000001
1,1,0.008069
2,2,0.061746
3,3,0.001549
4,4,0.000004
5,5,0.000000
6,6,0.013778
7,7,0.000000
8,8,0.008388
9,9,0.001645


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# SARSA

In [30]:
# Now, let's define the epsilon-greedy policy. We generate a random number from
# the uniform distribution and if the random number is less than epsilon, we select the
# random action, else we select the best action that has the maximum Q value:
def epsilon_greedy(state, epsilon):
    if random.uniform(0,1) < epsilon:
        return env.action_space.sample()
    else:
        return max(list(range(env.action_space.n)), key = lambda x:Q[(state,x)])

In [31]:
#Let's define the dictionary for storing the Q value of the state-action pair and
#initialize the Q value of all the state-action pairs to 0.0:
Q = {}
for s in range(env.observation_space.n):
    for a in range(env.action_space.n):
        Q[(s,a)] = 0.0

In [32]:
#Compute the policy
#For each episode:
for i in range(num_episodes):
#Initialize the state by resetting the environment:
    s = env.reset()
#Select the action using the epsilon-greedy policy:
    a = epsilon_greedy(s,epsilon)
#For each step in the episode:
    for t in range(num_timesteps):
#Perform the selected action and store the next state information:
        s_, r, done, _ = env.step(a)
#Select the action 𝑎′ in the next state 𝑠′using the epsilon-greedy policy:
        a_ = epsilon_greedy(s_,epsilon)
#Compute the Q value of the state-action pair as
        Q[(s,a)] += alpha * (r + gamma * Q[(s_,a_)]-Q[(s,a)])
        #print(Q[(s,a)])
# Update s=s' (update the next state s' to the current state s)
        s = s_
        a = a_
#If the current state is the terminal state, then break:
        if done:
            break

In [33]:
df = pd.DataFrame(list(Q.items()), columns=['state', 'value'])

In [34]:
df

,state,value
0,"(0, 0)",0.000017
1,"(0, 1)",0.000013
2,"(0, 2)",0.000017
3,"(0, 3)",0.000009
4,"(1, 0)",0.000001
...,...,...
59,"(14, 3)",0.129593
60,"(15, 0)",0.000000
61,"(15, 1)",0.000000
62,"(15, 2)",0.000000


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Q Learning

In [35]:
import numpy as np

In [36]:
#Compute the policy
#For each episode:
for i in range(num_episodes):
#Initialize the state by resetting the environment:
    s = env.reset()
#For each step in the episode:
    for t in range(num_timesteps):
#Select the action using the epsilon-greedy policy:
        a = epsilon_greedy(s,epsilon)
#Perform the selected action and store the next state information:
        s_, r, done, _ = env.step(a)
#Select the action 𝑎′ that has the maximum Q value in the next state 𝑠′:
        a_ = np.argmax([Q[(s_,a)] for a in range(env.action_space.n)])
#Compute the Q value of the state-action pair as
        Q[(s,a)] += alpha * (r + gamma * Q[(s_,a_)]-Q[(s,a)])
        #print(Q[(s,a)])
# Update s=s' (update the next state s' to the current state s)
        s = s_
        #a = a_
#If the current state is the terminal state, then break:
        if done:
            break

In [37]:
df = pd.DataFrame(list(Q.items()), columns=['state', 'value'])

In [38]:
df



,state,value
0,"(0, 0)",0.352488
1,"(0, 1)",0.309509
2,"(0, 2)",0.332866
3,"(0, 3)",0.325053
4,"(1, 0)",0.197165
...,...,...
59,"(14, 3)",0.936660
60,"(15, 0)",0.000000
61,"(15, 1)",0.000000
62,"(15, 2)",0.000000


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
